# 自训练：让模型自己给自己"标注"数据
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：04_半监督学习 | 源文件：自训练.py | 核心功能：SelfTrainingClassifier、伪标签策略、手动自训练实现

## 概述

自训练（Self-Training）是最直觉的半监督学习方法：用少量有标签数据训练一个初始模型，然后用这个模型对无标签数据做预测。高置信度的预测被当作"伪标签"加入训练集，模型重新训练。如此迭代，直到没有新的高置信度样本。

这种方法简单到可以手动实现，但在实践中效果出奇地好——前提是初始模型已经有合理的准确率。

脚本演示了 sklearn 的 SelfTrainingClassifier，以及一个手动实现的自训练流程，方便理解每一步的逻辑。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
# --- 导入库 ---
from sklearn.semi_supervised import SelfTrainingClassifier

## 数学原理

### 1. 半监督学习的理论框架

**代码对应**：`SelfTrainingClassifier` 使用少量有标签数据和大量无标签数据训练。

半监督学习的核心假设：

**聚类假设**（Cluster Assumption）：同一聚类中的样本倾向于属于同一类别。

**流形假设**（Manifold Assumption）：数据分布在低维流形上，流形上相邻的点具有相似的标签。

### 2. 自训练算法的数学描述

设 $L = \{(\mathbf{x}_i, y_i)\}_{i=1}^{l}$ 为有标签数据，$U = \{\mathbf{x}_j\}_{j=l+1}^{n}$ 为无标签数据。

迭代过程：
1. 用 $L$ 训练模型 $f$
2. 对每个 $\mathbf{x}_j \in U$，计算预测概率 $P(y|\mathbf{x}_j) = f(\mathbf{x}_j)$
3. 选取高置信度样本：$\hat{L} = \{(\mathbf{x}_j, \hat{y}_j) : \max_c P(y=c|\mathbf{x}_j) \geq \tau\}$
4. 更新 $L \leftarrow L \cup \hat{L}$，$U \leftarrow U \setminus \hat{L}$
5. 重复直到无新样本加入

**代码对应**：`threshold=0.8` 控制置信度阈值 $\tau$。

### 3. 确认偏差与缓解

自训练的风险是**确认偏差**（Confirmation Bias）：如果初始模型对某些无标签样本给出错误但高置信的预测，这些错误伪标签会被加入训练集并传播。

缓解策略：
- 提高阈值 $\tau$（更保守）
- 使用集成模型（多个模型投票）
- 逐步增加阈值（渐进式自训练）

### 1. 加载并准备数据

运行 1. 加载并准备数据 的代码，观察算法在该环节的行为。

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 只保留少量标签（约 10%），其余标记为 -1（未标记）
np.random.seed(42)
n_labeled = int(len(y_train_full) * 0.1)  # 仅 10% 有标签
labeled_idx = np.random.choice(len(y_train_full), n_labeled, replace=False)
y_train_semi = np.full_like(y_train_full, fill_value=-1)
y_train_semi[labeled_idx] = y_train_full[labeled_idx]

print(f"=== 数据划分 ===")
print(f"训练集: {len(X_train_full)} 样本 (有标签: {(y_train_semi >= 0).sum()}, "
      f"无标签: {(y_train_semi == -1).sum()})")
print(f"测试集: {len(X_test)} 样本")

### 2. 有标签数据训练（基准）

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
lr_labeled = RandomForestClassifier(n_estimators=100, random_state=42)
lr_labeled.fit(X_train_full[labeled_idx], y_train_full[labeled_idx])
acc_labeled = lr_labeled.score(X_test, y_test)
print(f"\n=== 仅用有标签数据训练 ===")
print(f"测试准确率: {acc_labeled:.4f}")

### 3. 全部标签训练（上限）

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
lr_full = RandomForestClassifier(n_estimators=100, random_state=42)
lr_full.fit(X_train_full, y_train_full)
acc_full = lr_full.score(X_test, y_test)
print(f"\n=== 全部标签训练（理想上限）===")
print(f"测试准确率: {acc_full:.4f}")

### 4. SelfTrainingClassifier

运行 4. SelfTrainingClassifier 的代码，观察算法在该环节的行为。

In [ ]:
# threshold: 伪标签的最低置信度阈值
# max_iter: 最大迭代次数
# criterion: 选择伪标签的策略 ("threshold" 或 "k_best")
st = SelfTrainingClassifier(
    estimator=RandomForestClassifier(n_estimators=100, random_state=42),
    threshold=0.8,
    max_iter=20,
    criterion="threshold",
# --- 继续 ---
)
st.fit(X_train_full, y_train_semi)

acc_semi = st.score(X_test, y_test)
print(f"\n=== 自训练 (Self-Training) ===")
print(f"测试准确率: {acc_semi:.4f}")
print(f"迭代次数: {st.n_iter_}")
print(f"最终有标签样本数: {len(st.labeled_iter_)}")
# labeled_iter_ 记录每轮新增的伪标签

### 5. 不同置信度阈值

运行 5. 不同置信度阈值 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 不同阈值 (threshold) 对比 ===")
for thresh in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    st_t = SelfTrainingClassifier(
        estimator=RandomForestClassifier(n_estimators=100, random_state=42),
        threshold=thresh, max_iter=20,
# --- 继续 ---
    )
    st_t.fit(X_train_full, y_train_semi)
    acc = st_t.score(X_test, y_test)
    print(f"  threshold={thresh}: 测试准确率={acc:.4f}, 迭代={st_t.n_iter_}")

### 6. 手动实现简单自训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== 手动自训练流程 ===")
base_clf = RandomForestClassifier(n_estimators=100, random_state=42)
X_labeled = X_train_full[labeled_idx].copy()
y_labeled = y_train_full[labeled_idx].copy()
X_unlabeled = np.delete(X_train_full, labeled_idx, axis=0)
# --- 数值计算 ---
y_unlabeled_true = np.delete(y_train_full, labeled_idx)

for iteration in range(5):
    # 1. 用当前有标签数据训练
    base_clf.fit(X_labeled, y_labeled)
    # 2. 对无标签数据预测
    probs = base_clf.predict_proba(X_unlabeled)
    max_probs = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    # 3. 选取高置信度预测作为伪标签
    high_conf = max_probs >= 0.9
    if high_conf.sum() == 0:
        print(f"  迭代 {iteration+1}: 无高置信度样本，停止")
        break
    # 4. 加入训练集
    X_labeled = np.vstack([X_labeled, X_unlabeled[high_conf]])
    y_labeled = np.concatenate([y_labeled, preds[high_conf]])
    X_unlabeled = X_unlabeled[~high_conf]
    print(f"  迭代 {iteration+1}: 新增 {high_conf.sum()} 个伪标签, "
          f"总标签数={len(y_labeled)}, 剩余无标签={len(X_unlabeled)}")

# 最终评估
acc_manual = base_clf.score(X_test, y_test)
print(f"  手动自训练测试准确率: {acc_manual:.4f}")

print("\n=== 自训练要点 ===")
print("- 优点：简单直观，不需要特殊架构")
print("- 缺点：初始模型差时，伪标签错误会传播（确认偏差）")
print("- threshold 太低：引入错误标签；太高：几乎不新增样本")
print("- 适合：初始模型已有一定准确率的场景")
# --- 输出结果 ---
print("- 与全监督对比，半监督可显著提升标签不足时的性能")

## 关键代码解释

### threshold——最关键的参数

```python
st = SelfTrainingClassifier(
    estimator=RandomForestClassifier(n_estimators=100),
    threshold=0.8, max_iter=20
)
```

threshold=0.8 意味着只有预测概率 >= 0.8 的样本才会被赋予伪标签。阈值太低会引入错误标签（确认偏差），阈值太高则几乎不新增样本。

### 手动实现的四步循环

```python
base_clf.fit(X_labeled, y_labeled)           # 1. 训练
probs = base_clf.predict_proba(X_unlabeled)  # 2. 预测
high_conf = max_probs >= 0.9                 # 3. 筛选
X_labeled = vstack([X_labeled, ...])         # 4. 扩充训练集
```

## 使用示例

```python
from sklearn.semi_supervised import SelfTrainingClassifier
st = SelfTrainingClassifier(estimator=RandomForestClassifier(), threshold=0.8)
st.fit(X, y_semi)  # y_semi 中 -1 表示无标签
```

## 注意事项

1. **确认偏差**：初始模型的错误会被伪标签"固化"并放大
2. **阈值选择**：通常 0.7-0.95，根据初始模型的置信度校准
3. **适用场景**：初始模型已有合理准确率（> 60%）
4. **不适用场景**：初始模型太差时，自训练可能越训越差

## 总结与延伸

以上代码展示了 **自训练** 的完整流程。

**解读要点**：
- 关注 **标签传播** 的准确率随迭代的变化
- 对比有监督 vs 半监督的性能差距
- 观察少量标签下的学习效果

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Noisy Student**：Google 的自训练变体，在 ImageNet 上刷新了记录
- **伪标签 + 数据增强**：FixMatch 等方法结合了自训练和一致性正则化
- **Tri-Training**：用三个模型互相标注，减少单模型的偏差
- **自训练与主动学习的结合**：用自训练标注高置信度样本，用主动学习标注高不确定性样本
